# 02 - Chunking Lab

Five chunking strategies, two evaluation layers, one question: does
structure-aware chunking beat a plain character splitter on financial filings?

Logic lives in `src/chunking/*` and `src/eval/*`. The long-running sweeps have
script equivalents: `scripts/build_chunking_strategies.py`,
`scripts/run_retrieval_eval.py`, `scripts/run_generation_eval.py`.


In [ ]:
# notebooks are thin: every function called here is imported from src/
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.chunking import build_all
from src.chunking.naive_fixed import naive_fixed_size_chunks
from src.chunking.row_level import build_row_level_chunks
from src.chunking.sentence_window import build_sentence_window_chunks
from src.chunking.whole_table import build_whole_table_chunks
from src.utils.io import load_jsonl, save_jsonl

documents = load_jsonl("data/processed/documents.jsonl")
eval_examples = load_jsonl("data/processed/eval_dataset.jsonl")
print(f"{len(documents)} documents, {len(eval_examples)} questions")


## 1. Build the chunk stores

| # | strategy | idea |
|---|----------|------|
| 1 | row-level | one chunk per table row / text line (gold-aligned baseline) |
| 2 | naive fixed-size | 500-char windows over flat text, structure-blind control |
| 3 | whole-table | the entire table as one chunk, text unchanged from 1 |
| 4 | sentence-window | text line embedded with its neighbours, table rows unchanged from 1 |
| 5 | parent-child | *planned* - match on the row, return the parent table |


In [ ]:
strategies = {
    "row_level": (build_row_level_chunks, "data/processed/chunks.jsonl"),
    "naive_fixed": (naive_fixed_size_chunks, "data/processed/chunks_naive_fixed.jsonl"),
    "whole_table": (build_whole_table_chunks, "data/processed/chunks_whole_table.jsonl"),
    "sentence_window": (build_sentence_window_chunks, "data/processed/chunks_sentence_window.jsonl"),
}

for name, (builder, out_path) in strategies.items():
    chunks = build_all(documents, builder)
    save_jsonl(chunks, out_path)
    print(f"{name:<16} {len(chunks):>6} chunks")


In [ ]:
# what the granularity difference actually looks like
whole_table = build_all(documents[:1], build_whole_table_chunks)
row_level = build_all(documents[:1], build_row_level_chunks)

print("row-level table chunk:")
print(" ", next(c["text"] for c in row_level if c["chunk_type"] == "table_row")[:220])
print("\nwhole-table chunk:")
print(" ", next(c["text"] for c in whole_table if c["chunk_type"] == "whole_table")[:220].replace("\n", "\n  "))


## 2. Retrieval eval

Two scorers, and picking the wrong one invalidates the comparison:

- `evaluate_chunking_strategy` matches gold ids exactly. Only valid for the
  row-level id scheme - a `whole_table` chunk can never equal a `table_row` id.
- `evaluate_chunking_strategy_generic` matches on content overlap, so it works
  across granularities. This is the one used for cross-strategy numbers.

Noise-flagged chunks are excluded from every index via `load_index_chunks`.


In [ ]:
from src.eval.retrieval_harness import (
    evaluate_chunking_strategy,
    evaluate_chunking_strategy_generic,
    gold_text_index,
    load_embedder,
    load_index_chunks,
)
from src.utils.io import save_json

model = load_embedder()
chunk_id_to_gold_text = gold_text_index(load_jsonl("data/processed/chunks.jsonl"))

result_paths = {
    "row_level": "data/processed/eval_results_baseline_row_level.json",
    "naive_fixed": "data/processed/eval_results_naive_fixed.json",
    "whole_table": "data/processed/eval_results_whole_table.json",
    "sentence_window": "data/processed/eval_results_sentence_window.json",
}

for name, (_, chunks_path) in strategies.items():
    chunks = load_index_chunks(chunks_path)
    print(f"\nevaluating {name}: {len(chunks)} chunks...")
    summary = evaluate_chunking_strategy_generic(chunks, eval_examples, chunk_id_to_gold_text, model)
    for k, s in summary.items():
        print(f"  k={k:>2}  precision={s['precision']:.3f}  recall={s['recall']:.3f}")
    save_json(summary, result_paths[name])


In [ ]:
# exact-id scoring, shown only for the baseline where it is meaningful
exact = evaluate_chunking_strategy(load_index_chunks("data/processed/chunks.jsonl"),
                                    eval_examples, model)
for k, s in exact.items():
    print(f"k={k:>2}  precision={s['precision']:.3f}  recall={s['recall']:.3f}")


## 3. Generation eval

Retrieval metrics only say whether the evidence was fetched. What settles the
argument is whether the LLM could then compute the answer, so each strategy is
run through a local Ollama model on a fixed, operation-stratified 44-question
sample. Ground truth is `exe_ans`, not the noisy human-typed `answer` field.

Roughly 4 x 44 sequential LLM calls, so this cell takes 15-20 minutes.


In [ ]:
from src.eval.generation_harness import LLM_MODEL, run_generation_eval
from src.eval.sampling import stratified_sample_by_op

sample = stratified_sample_by_op(eval_examples, per_op_n=5)
save_jsonl(sample, "data/processed/gen_eval_sample.jsonl")
print(f"{len(sample)} questions, LLM={LLM_MODEL}")

all_results = {}
for name, (_, chunks_path) in strategies.items():
    print(f"\n=== {name} ===")
    result = run_generation_eval(load_index_chunks(chunks_path), sample, model, top_k=5)
    print(f"{name} accuracy: {result['accuracy']:.1%}")
    all_results[name] = result

save_json({k: v["accuracy"] for k, v in all_results.items()},
          "data/processed/generation_comparison.json")
save_json(all_results, "data/processed/eval_results_generation.json")


## 4. Comparison

Note the disagreement between the two layers: row-level wins on retrieval
precision but whole-table wins on answer accuracy. Most FinQA questions need two
cells from the same table, and a whole-table chunk hands the model both at once.


In [ ]:
from scripts.build_comparison_report import build_report

retrieval, generation = build_report(show=True)


## 5. Strategy 5: parent-child (small-to-big)

**Motivation:** row-level is precise but misses context; whole-table has context but dilutes precision.
**Idea:** retrieve on row-level children, then expand the context handed to the LLM to the whole
parent table (or sentence-window for text lines). The retrieval index is byte-identical to strategy 1,
so `retrieval_hit_rate` IS comparable here.

The map is built once from `documents` and stays in memory for all queries.


In [ ]:
from src.chunking.parent_child import build_index_chunks, build_parent_text_map
from src.utils.io import save_jsonl

# index chunks = row-level (same ids as strategy 1, retrieval scores are comparable)
pc_index_chunks = build_index_chunks(documents)
save_jsonl(pc_index_chunks, "data/processed/chunks_parent_child_index.jsonl")
print(f"parent-child index: {len(pc_index_chunks)} chunks (identical to row-level baseline)")

# parent map: child_id -> whole-table text (or sentence-window for text lines)
parent_text_map = build_parent_text_map(documents)
print(f"parent map: {len(parent_text_map)} child->parent entries")

# spot-check: what does the first question's gold child expand to?
first_ex = eval_examples[0]
first_gold_id = first_ex["gold_chunk_ids"][0]
print(f"\ngold child: {first_gold_id}")
print(f"parent context (first 300 chars):\n{parent_text_map.get(first_gold_id, 'NOT FOUND')[:300]}")


### 5b. Parent-child generation eval

Retrieval only, no LLM: not very meaningful here since it's the same index as strategy 1.
The real test is generation accuracy -- does giving the LLM the full parent table help?

This cell calls Ollama (~44 questions, budget ~5 minutes for one strategy).


In [ ]:
from src.eval.generation_harness import LLM_MODEL, answers_match, run_parent_child_eval
from src.utils.io import load_jsonl, save_json

sample = load_jsonl("data/processed/gen_eval_sample.jsonl")
print(f"evaluating parent-child on {len(sample)} questions, LLM={LLM_MODEL}")

pc_chunks = [c for c in pc_index_chunks if not c.get("is_noise", False)]
pc_result = run_parent_child_eval(pc_chunks, parent_text_map, sample, model, top_k=5)

print(f"\nparent-child accuracy:  {pc_result['accuracy']:.1%} ({sum(r['correct'] for r in pc_result['results'])}/{pc_result['n']})")
print(f"retrieval_hit_rate:     {pc_result['retrieval_hit_rate']:.1%}")

save_json(pc_result, "data/processed/eval_results_parent_child.json")


## 6. Table chunk-size sweep (retrieval only)

Strategies 1 and 3 are the two ends of a single dial. `src/chunking/row_group.py` makes
the group size a parameter, so the space between them can be measured.
No Ollama needed — retrieval only, ~2 minutes.


In [ ]:
from scripts.run_chunk_size_sweep import run_sweep

sweep = run_sweep(sizes=[1, 2, 3, 5, 999], top_k=5, model=model)


## 7. Known caveats

- `gold_hit` / `retrieval_hit_rate` in `run_generation_eval` uses exact chunk_id matching,
  so it is **not** comparable across strategies with different id schemes (whole_table vs table_row).
  Parent-child uses the row-level ids, so its hit_rate IS comparable to strategy 1.
- 65 of 883 questions carry no gold annotations in FinQA; those are excluded from recall denominators.
- All numbers are from a 44-question sample — interpret narrow margins as suggestive, not settled.
